# ScreenTrans — server dịch trên Kaggle (Phase 5)

Chạy một LLM ~7B (lượng tử 4-bit) trên GPU miễn phí của Kaggle và mở nó ra ngoài qua ngrok, để app ScreenTrans
dùng làm engine **Kaggle** (Cài đặt → Dịch bằng → Kaggle).

1. Settings của notebook: **Accelerator = GPU T4 x2** (hoặc P100), **Internet = On**.
2. Add-ons → Secrets: thêm `NGROK_TOKEN` (token miễn phí ở dashboard.ngrok.com).
3. Run all. Ô cuối in ra **URL** và **token**: dán URL vào "URL server Kaggle", token vào "Key / token" trong app.

App gửi `POST /translate` `{"system": ..., "prompt": ..., "source_lang": "en"}` (câu lệnh đã dựng sẵn, giống các
engine khác) với `Authorization: Bearer <token>`; server trả `{"translations": [...]}`. `GET /health` cho biết model.
Phiên Kaggle tối đa 12 giờ; tắt notebook là server dừng (app tự dùng model trên máy).

In [ ]:
!pip install -q vllm fastapi uvicorn pyngrok

In [ ]:
import secrets

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # nhanh hơn: "Qwen/Qwen2.5-3B-Instruct-AWQ"
TOKEN = secrets.token_urlsafe(24)       # app gửi token này; người khác có URL cũng không dùng được
PORT = 8000

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL, quantization="awq", max_model_len=4096, gpu_memory_utilization=0.90,
          tensor_parallel_size=1)
params = SamplingParams(temperature=0.2, max_tokens=1024)
print(llm.chat([{"role": "user", "content": "Say OK."}], params)[0].outputs[0].text)

In [ ]:
import json, re, threading
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn

app = FastAPI()
lock = threading.Lock()  # one generation at a time (one GPU)


class Req(BaseModel):
    system: str
    prompt: str
    source_lang: str = "en"


def translations(text: str):
    m = re.search(r"[\[{]", text)
    if not m:
        return None
    try:
        data, _ = json.JSONDecoder().raw_decode(text, m.start())
    except ValueError:
        return None
    items = data.get("translations") if isinstance(data, dict) else data
    return items if isinstance(items, list) else None


@app.get("/health")
def health():
    return {"model": MODEL, "languages": ["en"]}


@app.post("/translate")
def translate(req: Req, authorization: str = Header("")):
    if authorization != f"Bearer {TOKEN}":
        raise HTTPException(401, "bad token")
    with lock:
        out = llm.chat([{"role": "system", "content": req.system}, {"role": "user", "content": req.prompt}],
                       params)[0].outputs[0].text
    items = translations(out)
    if items is None:
        raise HTTPException(502, "model reply was not JSON")
    return {"translations": items}


threading.Thread(target=uvicorn.run, args=(app,), kwargs={"host": "0.0.0.0", "port": PORT, "log_level": "warning"},
                 daemon=True).start()

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
public = ngrok.connect(PORT, "http").public_url
print("URL server Kaggle:", public)
print("Key / token:     ", TOKEN)